# Notebook 05 — DiCE Counterfactual Explanations

Generates actionable "what-if" scenarios for rejected applicants
showing the minimum changes needed to get approved.

In [1]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import dice_ml
from dice_ml import Dice

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

In [2]:
# Load original (unscaled) data — DiCE needs real feature ranges
X_orig = pd.read_csv('../outputs/X_original.csv')
y_orig = pd.read_csv('../outputs/y_original.csv')['loan_approved']

df_full = X_orig.copy()
df_full['loan_approved'] = y_orig
print(f'Shape: {df_full.shape}')
df_full.head()

Shape: (12367, 15)


,age,income,assets,credit_score,debt_to_income_ratio,existing_loan,criminal_record,credit_tier,high_debt,risk_flags,asset_income_ratio,age_group,income_per_age,financial_health,loan_approved
0,56.0,53779.0,771450.000000,700.0,0.45,1,0,2,1,2,14.344552,2,943.491228,0.644412,0
1,65.0,141745.0,525485.720059,889.0,0.34,1,1,4,0,2,3.707235,3,2147.651515,0.616353,0
2,60.0,37340.0,459420.000000,788.0,0.35,0,1,3,0,1,12.303366,2,612.131148,0.715824,0
3,61.0,101388.0,168102.000000,831.0,0.32,1,1,4,0,2,1.657991,3,1635.290323,0.595059,0
4,27.0,83269.0,187546.000000,691.0,0.21,0,0,2,0,0,2.252264,0,2973.892857,0.862176,0


In [3]:
# Train a light RF on unscaled data for DiCE
X = df_full.drop('loan_approved', axis=1)
y = df_full['loan_approved']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

dice_rf = RandomForestClassifier(n_estimators=200, max_depth=15,
                                  min_samples_split=5, random_state=42, n_jobs=-1)
dice_rf.fit(Xtr, ytr)
yp = dice_rf.predict(Xte)
print(f'DiCE RF — Accuracy: {accuracy_score(yte, yp):.4f}  F1: {f1_score(yte, yp):.4f}')

DiCE RF — Accuracy: 0.9996  F1: 0.9982


In [4]:
continuous = ['age','income','assets','credit_score','debt_to_income_ratio',
              'asset_income_ratio','income_per_age','financial_health']
continuous = [c for c in continuous if c in X.columns]

d_data = dice_ml.Data(dataframe=df_full, continuous_features=continuous, outcome_name='loan_approved')
d_model = dice_ml.Model(model=dice_rf, backend='sklearn', model_type='classifier')
dice_exp = Dice(d_data, d_model, method='random')
print('DiCE explainer ready')

DiCE explainer ready


In [5]:
# Generate counterfactuals for 5 rejected applicants
rejected_idx = Xte[yp == 0].index.tolist()[:5]
cf_results = []

for i, idx in enumerate(rejected_idx):
    row = Xte.loc[idx]
    print(f'\n{"="*65}\nApplicant {i+1} (idx={idx}) — REJECTED\n{"="*65}')
    for c in X.columns:
        print(f'  {c:<25} = {row[c]:.4f}' if isinstance(row[c], float) else f'  {c:<25} = {row[c]}')
    query = pd.DataFrame([row], columns=X.columns)
    try:
        res = dice_exp.generate_counterfactuals(
            query, total_CFs=3, desired_class='opposite',
            features_to_vary=continuous, random_seed=42+i)
        cf_df = res.cf_examples_list[0].final_cfs_df
        if cf_df is not None and len(cf_df) > 0:
            print('\nCounterfactual changes for APPROVAL:')
            for ci, cr in cf_df.iterrows():
                print(f'  Scenario {ci+1}:')
                for c in X.columns:
                    if abs(float(cr[c]) - float(row[c])) > 0.001:
                        d = 'UP' if float(cr[c]) > float(row[c]) else 'DOWN'
                        print(f'    {c}: {row[c]:.2f} -> {cr[c]:.2f} ({d})')
            cf_results.append({'idx': idx, 'original': row.to_dict(), 'cfs': cf_df.to_dict('records')})
    except Exception as e:
        print(f'  Could not generate CFs: {e}')


Applicant 1 (idx=11775) — REJECTED
  age                       = 44.0000
  income                    = 132607.0000
  assets                    = 229007.0000
  credit_score              = 727.0000
  debt_to_income_ratio      = 0.4500
  existing_loan             = 1.0000
  criminal_record           = 1.0000
  credit_tier               = 2.0000
  high_debt                 = 1.0000
  risk_flags                = 3.0000
  asset_income_ratio        = 1.7269
  age_group                 = 1.0000
  income_per_age            = 2946.8222
  financial_health          = 0.5071


  0%|                                                     | 0/1 [00:00<?, ?it/s]

  0%|                                                     | 0/1 [00:00<?, ?it/s]

  Could not generate CFs: ('Feature', 'existing_loan', 'has a value outside the dataset.')

Applicant 2 (idx=7000) — REJECTED
  age                       = 31.0000
  income                    = 47100.0000
  assets                    = 734391.0000
  credit_score              = 713.0000
  debt_to_income_ratio      = 0.3100
  existing_loan             = 0.0000
  criminal_record           = 0.0000
  credit_tier               = 2.0000
  high_debt                 = 0.0000
  risk_flags                = 0.0000
  asset_income_ratio        = 15.5918
  age_group                 = 1.0000
  income_per_age            = 1471.8750
  financial_health          = 0.8425


  0%|                                                     | 0/1 [00:00<?, ?it/s]

...

  0%|                                                     | 0/1 [00:00<?, ?it/s]

  Could not generate CFs: ('Feature', 'existing_loan', 'has a value outside the dataset.')

Applicant 5 (idx=6891) — REJECTED
  age                       = 46.0000
  income                    = 63658.0000
  assets                    = 930676.0000
  credit_score              = 861.0000
  debt_to_income_ratio      = 0.2500
  existing_loan             = 1.0000
  criminal_record           = 1.0000
  credit_tier               = 4.0000
  high_debt                 = 0.0000
  risk_flags                = 2.0000
  asset_income_ratio        = 14.6197
  age_group                 = 2.0000
  income_per_age            = 1354.4255
  financial_health          = 0.6302


  0%|                                                     | 0/1 [00:00<?, ?it/s]

  0%|                                                     | 0/1 [00:00<?, ?it/s]

  Could not generate CFs: ('Feature', 'existing_loan', 'has a value outside the dataset.')


In [6]:
# Visualise which features change most
if cf_results:
    changes = {c: 0 for c in X.columns}
    total = 0
    for r in cf_results:
        for cf in r['cfs']:
            total += 1
            for c in X.columns:
                if c in cf and abs(float(cf[c]) - float(r['original'][c])) > 0.001:
                    changes[c] += 1
    ch = pd.DataFrame({'Feature': list(changes.keys()), 'Pct': [v/max(total,1)*100 for v in changes.values()]})
    ch = ch[ch['Pct'] > 0].sort_values('Pct')
    plt.figure(figsize=(12, 6))
    plt.barh(ch['Feature'], ch['Pct'], color='coral', edgecolor='black')
    plt.xlabel('% of counterfactuals where feature changed')
    plt.title('Most Frequently Changed Features', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.savefig('../outputs/dice_feature_changes.png', dpi=150, bbox_inches='tight'); plt.show()

In [7]:
# Side-by-side bar for first result
if cf_results:
    r = cf_results[0]; o = r['original']; c = r['cfs'][0]
    feats = [f for f in continuous if abs(float(c[f]) - float(o[f])) > 0.001]
    if feats:
        x = np.arange(len(feats)); w = 0.35
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.bar(x - w/2, [o[f] for f in feats], w, label='Original (Rejected)', color='#e74c3c', edgecolor='black')
        ax.bar(x + w/2, [c[f] for f in feats], w, label='Counterfactual (Approved)', color='#27ae60', edgecolor='black')
        ax.set_xticks(x); ax.set_xticklabels(feats, rotation=45, ha='right')
        ax.legend(); ax.set_title('Original vs Counterfactual', fontsize=14, fontweight='bold')
        plt.tight_layout(); plt.savefig('../outputs/dice_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

print('\nDiCE analysis complete.')


DiCE analysis complete.
